# Making the SVM model work

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.base import BaseEstimator, TransformerMixin

In [4]:
# Resolve dataset path robustly from the current notebook working directory.
cwd = Path.cwd()
candidate_paths = [
    cwd / "datasets" / "nyc-taxi-trip-duration",
    cwd.parent / "datasets" / "nyc-taxi-trip-duration",
    cwd.parent.parent / "datasets" / "nyc-taxi-trip-duration",
    cwd / "chapters" / "chapter_2" / "datasets" / "nyc-taxi-trip-duration",
]

base_path = next((p for p in candidate_paths if p.exists()), None)
if base_path is None:
    raise FileNotFoundError(
        "Could not find 'nyc-taxi-trip-duration'. "
        f"Current working dir: {cwd}. Tried: {candidate_paths}"
    )

train_path = base_path / "train" / "train.csv"
test_path = base_path / "test" / "test.csv"
train_df = pd.read_csv(train_path).drop(columns=["dropoff_datetime"], errors="ignore")
test_df = pd.read_csv(test_path)
print(f"Working dir: {cwd}")
print(f"Loaded: {train_path} -> {train_df.shape}")
print(f"Loaded: {test_path} -> {test_df.shape}")
display(train_df.head())
display(test_df.head())

Working dir: c:\Users\marco\Documents\code_classes\machine_learning_review\chapters\chapter_2\exercises
Loaded: c:\Users\marco\Documents\code_classes\machine_learning_review\chapters\chapter_2\datasets\nyc-taxi-trip-duration\train\train.csv -> (1458644, 10)
Loaded: c:\Users\marco\Documents\code_classes\machine_learning_review\chapters\chapter_2\datasets\nyc-taxi-trip-duration\test\test.csv -> (625134, 9)


,id,vendor_id,pickup_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag,trip_duration
0,id2875421,2,2016-03-14 17:24:55,1,-73.982155,40.767937,-73.964630,40.765602,N,455
1,id2377394,1,2016-06-12 00:43:35,1,-73.980415,40.738564,-73.999481,40.731152,N,663
2,id3858529,2,2016-01-19 11:35:24,1,-73.979027,40.763939,-74.005333,40.710087,N,2124
3,id3504673,2,2016-04-06 19:32:31,1,-74.010040,40.719971,-74.012268,40.706718,N,429
4,id2181028,2,2016-03-26 13:30:55,1,-73.973053,40.793209,-73.972923,40.782520,N,435


,id,vendor_id,pickup_datetime,passenger_count,pickup_longitude,pickup_latitude,dropoff_longitude,dropoff_latitude,store_and_fwd_flag
0,id3004672,1,2016-06-30 23:59:58,1,-73.988129,40.732029,-73.990173,40.756680,N
1,id3505355,1,2016-06-30 23:59:53,1,-73.964203,40.679993,-73.959808,40.655403,N
2,id1217141,1,2016-06-30 23:59:47,1,-73.997437,40.737583,-73.986160,40.729523,N
3,id2150126,2,2016-06-30 23:59:41,1,-73.956070,40.771900,-73.986427,40.730469,N
4,id1598245,1,2016-06-30 23:59:33,1,-73.970215,40.761475,-73.961510,40.755890,N


In [5]:
class DataProcessor:
    def __init__(self, df_train, df_test):
        self.df_train = df_train
        self.df_test = df_test

    def detect_missing_values(self):
        missing_train = self.df_train.isnull().sum()
        missing_test = self.df_test.isnull().sum()
        print("Missing values in training set:")
        print(missing_train[missing_train > 0])
        print("\nMissing values in test set:")
        print(missing_test[missing_test > 0])

    def clear_null(self):
        self.df_train = self.df_train.dropna()
        self.df_test = self.df_test.dropna()

    def clear_nan(self):
        self.df_train = self.df_train.replace([np.inf, -np.inf], np.nan).dropna()
        self.df_test = self.df_test.replace([np.inf, -np.inf], np.nan).dropna()

    def run_all(self):
        self.detect_missing_values()
        self.clear_null()
        self.clear_nan()

In [6]:
class ValuationModel:
    def __init__(self, df_test):
        self.df_test_fe = df_test

    def mae(self, y_true, y_pred):
        return np.mean(np.abs(y_true - y_pred))

    def rmse(self, y_true, y_pred):
        return np.sqrt(np.mean((y_true - y_pred) ** 2))

    def r2_score(self, y_true, y_pred):
        ss_res = np.sum((y_true - y_pred) ** 2)
        ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
        return 1 - (ss_res / ss_tot)

    def rmlse(self, y_true, y_pred):
        # Root Mean Logarithmic Error
        return np.sqrt(np.mean((np.log1p(y_true) - np.log1p(y_pred)) ** 2))

    def run_all(self, y_true, y_pred):
        print(f"MAE: {self.mae(y_true, y_pred):.4f}")
        print(f"RMSE: {self.rmse(y_true, y_pred):.4f}")
        print(f"R² Score: {self.r2_score(y_true, y_pred):.4f}")
        print(f"RMLSE: {self.rmlse(y_true, y_pred):.4f}")

In [7]:
from typing import Optional

import pandas as pd
from scipy.stats import loguniform
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.impute import SimpleImputer
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import FunctionTransformer, OneHotEncoder, RobustScaler
from sklearn.svm import LinearSVR


class FeatureEngineering(BaseEstimator, TransformerMixin):
    """Create temporal and rush-hour features for taxi data.

    This transformer learns the most expensive pickup hours on the training
    split and uses this information to generate the `is_rush_hour` feature
    during transformation.

    Args:
        top_k: Number of hours with highest mean trip duration to keep.
        min_rides: Minimum rides per hour required to consider the hour.
    """

    def __init__(self, top_k: int = 3, min_rides: int = 200) -> None:
        self.top_k = top_k
        self.min_rides = min_rides
        self.rush_hours: list[int] = []

    def fit(
        self, X: pd.DataFrame, y: Optional[pd.Series] = None
    ) -> "FeatureEngineering":
        """Learn rush-hour buckets from training data.

        Args:
            X: Input features dataframe.
            y: Target series with trip durations.

        Returns:
            The fitted transformer instance.
        """
        X_fit = X.copy()
        X_fit["pickup_datetime"] = pd.to_datetime(
            X_fit["pickup_datetime"], errors="coerce"
        )
        X_fit["pickup_hour"] = X_fit["pickup_datetime"].dt.hour

        if y is None:
            self.rush_hours = []
            return self

        temp = pd.DataFrame(
            {
                "pickup_hour": X_fit["pickup_hour"],
                "trip_duration": np.asarray(y),
            }
        ).dropna(subset=["pickup_hour", "trip_duration"])

        if temp.empty:
            self.rush_hours = []
            return self

        hour_stats = temp.groupby("pickup_hour", as_index=False).agg(
            mean_trip_duration=("trip_duration", "mean"),
            ride_count=("trip_duration", "size"),
        )

        hour_stats = hour_stats[hour_stats["ride_count"] >= self.min_rides]
        if hour_stats.empty:
            self.rush_hours = []
            return self

        self.rush_hours = (
            hour_stats.sort_values("mean_trip_duration", ascending=False)
            .head(self.top_k)["pickup_hour"]
            .astype(int)
            .tolist()
        )
        return self

    def transform(self, X: pd.DataFrame) -> pd.DataFrame:
        """Generate model-ready engineered features.

        Args:
            X: Raw feature dataframe.

        Returns:
            Dataframe with engineered temporal features.
        """
        X = X.copy()
        X["pickup_datetime"] = pd.to_datetime(X["pickup_datetime"], errors="coerce")
        X["pickup_hour"] = X["pickup_datetime"].dt.hour
        X["pickup_weekday_1_7"] = X["pickup_datetime"].dt.weekday + 1
        X["pickup_month"] = X["pickup_datetime"].dt.month
        X["pickup_day"] = X["pickup_datetime"].dt.day
        X["pickup_minute"] = X["pickup_datetime"].dt.minute

        X["pickup_hour_sin"] = np.sin(2 * np.pi * X["pickup_hour"] / 24)
        X["pickup_hour_cos"] = np.cos(2 * np.pi * X["pickup_hour"] / 24)
        X["is_weekend"] = (X["pickup_weekday_1_7"] >= 6).astype(int)
        X["is_rush_hour"] = X["pickup_hour"].isin(self.rush_hours).astype(int)

        return X.drop(columns=["pickup_datetime"], errors="ignore")


# Keep sample small for weak machines.
sample_n = min(6000, len(train_df))
train_sample = train_df.sample(n=sample_n, random_state=42).copy()

X = train_sample.drop(columns=["trip_duration"], errors="ignore")
y = train_sample["trip_duration"].copy()

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

numeric_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", RobustScaler()),
    ]
)

categorical_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    [
        ("num", numeric_pipeline, make_column_selector(dtype_include=np.number)),
        ("cat", categorical_pipeline, make_column_selector(dtype_exclude=np.number)),
    ]
)

# Drop store_and_fwd_flag inside sklearn Pipeline for both train and inference.
drop_store_and_fwd = FunctionTransformer(
    lambda df: df.drop(columns=["store_and_fwd_flag"], errors="ignore"),
    validate=False,
)

pipeline_model = Pipeline(
    [
        ("drop_store_and_fwd", drop_store_and_fwd),
        ("feature_engineering", FeatureEngineering()),
        ("preprocessor", preprocessor),
        ("model", LinearSVR(random_state=42, max_iter=5000)),
    ]
)

# Hyperparameters for LinearSVR (fast setup).
param_grid = {
    "feature_engineering__top_k": [1, 2, 3, 4],
    "feature_engineering__min_rides": [100, 200, 300],
    "model__C": loguniform(1e-2, 1e1),
    "model__epsilon": loguniform(1e-3, 1e-1),
    "model__loss": ["epsilon_insensitive", "squared_epsilon_insensitive"],
    "model__tol": [1e-3, 1e-4],
}

random_search = RandomizedSearchCV(
    estimator=pipeline_model,
    param_distributions=param_grid,
    n_iter=4,
    cv=8,
    scoring="neg_root_mean_squared_error",
    verbose=2,
    random_state=42,
    n_jobs=-1,
)

random_search.fit(X_train, y_train)
y_pred = np.clip(random_search.predict(X_valid), 0, None)

print("Best params:", random_search.best_params_)
print("Best CV score (neg RMSE):", random_search.best_score_)

valuator = ValuationModel(X_valid)
valuator.run_all(y_valid.to_numpy(), y_pred)

Fitting 8 folds for each of 4 candidates, totalling 32 fits
Best params: {'feature_engineering__min_rides': 300, 'feature_engineering__top_k': 4, 'model__C': np.float64(7.114476009343421), 'model__epsilon': np.float64(0.029106359131330698), 'model__loss': 'epsilon_insensitive', 'model__tol': 0.001}
Best CV score (neg RMSE): -2083.1786328985236
MAE: 553.6851
RMSE: 3503.3625
R² Score: -0.0045
RMLSE: 0.9270
